In [1]:
import os
os.chdir("../")
%pwd

'e:\\ml\\DataScience_Project1'

In [2]:
from dataclasses import dataclass
from pathlib import Path
import pandas as pd
import joblib

from sklearn.linear_model import ElasticNet
from src.datascience.utils.logger import logging
from src.datascience.utils.common import read_yaml, create_directories
from src.datascience.constants import *


In [7]:
# =========================
# CONFIG CLASS
# =========================
@dataclass
class ModelTrainerConfig:
    root_dir: Path
    training_data: Path
    testing_data: Path
    trained_model_file: Path
    alpha: float
    l1_ratio: float
    target_column: str



In [8]:
# CONFIGURATION MANAGER
# =========================
class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAMS_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet

        return ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            training_data=Path(config.training_data),
            testing_data=Path(config.testing_data),
            trained_model_file=Path(config.trained_model_file),
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=self.schema.TARGET_COLUMN.name
        )

In [11]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        # create directory
        Path(self.config.root_dir).mkdir(parents=True, exist_ok=True)

        # check files
        if not self.config.training_data.exists():
            raise FileNotFoundError(f"❌ Training file not found: {self.config.training_data}")

        if not self.config.testing_data.exists():
            raise FileNotFoundError(f"❌ Testing file not found: {self.config.testing_data}")

        logging.info("Reading training data")
        train_df = pd.read_csv(self.config.training_data)

        logging.info("Reading testing data")
        test_df = pd.read_csv(self.config.testing_data)

        # split
        X_train = train_df.drop(columns=[self.config.target_column])
        y_train = train_df[self.config.target_column]

        X_test = test_df.drop(columns=[self.config.target_column])
        y_test = test_df[self.config.target_column]

        # train model
        logging.info("Training ElasticNet model")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio)
        model.fit(X_train, y_train)

        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        import numpy as np
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Metrics
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        # Logging
        logging.info(f"MAE: {mae}")
        logging.info(f"MSE: {mse}")
        logging.info(f"RMSE: {rmse}")
        logging.info(f"R2 Score: {r2}")

        # save model
        joblib.dump(model, self.config.trained_model_file)

        logging.info(f"✅ Model saved at: {self.config.trained_model_file}")

In [12]:
# RUN
# =========================
try:
    config_manager = ConfigurationManager()
    config = config_manager.get_model_trainer_config()

    obj = ModelTrainer(config=config)
    obj.train_model()

except Exception as e:
    logging.error(f"❌ Error: {e}")
    raise e

[2026-03-20 23:36:57,707] 29 root - INFO - YAML file loaded successfully from: config\config.yaml
[2026-03-20 23:36:57,709] 29 root - INFO - YAML file loaded successfully from: params.yaml
[2026-03-20 23:36:57,713] 29 root - INFO - YAML file loaded successfully from: schema.yaml
[2026-03-20 23:36:57,715] 52 root - INFO - Created directory at: artifacts
[2026-03-20 23:36:57,719] 16 root - INFO - Reading training data
[2026-03-20 23:36:57,727] 19 root - INFO - Reading testing data
[2026-03-20 23:36:57,731] 30 root - INFO - Training ElasticNet model
[2026-03-20 23:36:57,738] 47 root - INFO - MAE: 0.5543829927884041
[2026-03-20 23:36:57,739] 48 root - INFO - MSE: 0.4801361292089501
[2026-03-20 23:36:57,740] 49 root - INFO - RMSE: 0.6929185588573523
[2026-03-20 23:36:57,742] 50 root - INFO - R2 Score: 0.2652917761622785
[2026-03-20 23:36:57,748] 55 root - INFO - ✅ Model saved at: artifacts\model_trainer\trained_model.joblib
